# VTKJS 三维模型

`PnVTK` 组件可以在 Panel 应用程序中渲染 vtk.js 文件，使得可以加载和交互复杂的 3D 几何体。

底层实现为`panel.pane.VTK`，参数基本一致，参考文档：https://panel.holoviz.org/reference/panes/VTKJS.html


In [1]:
##ignore
%load_ext vuepy
from panel_vuepy import vpanel


## 基本用法

构造 `PnVTKJS` 组件最简单的方法是给它一个 vtk.js 文件，它将序列化并嵌入到图表中。`PnVTKJS` 组件还支持 Bokeh 提供的常规尺寸选项，包括响应式尺寸模式。也可以通过替换 `object` 来更新模型。

In [2]:
%%vuepy_run --plugins vpanel --show-code
<template>
  <PnVTK
    :object="vtkjs_url.value" 
    sizing_mode="stretch_width" 
    :height="400" 
    :enable_keybindings="True" 
    :orientation_widget="True"
  />
  <PnButton @click="update_object()">更换 3D 模型</PnButton>
</template>
<script lang='py'>
from vuepy import ref

vtkjs_url = ref("https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs")

def update_object():
    vtkjs_url.value = "https://raw.githubusercontent.com/Kitware/vtk-js-datasets/master/data/vtkjs/TBarAssembly.vtkjs"
</script>

{"vue": "<!-- --plugins vpanel --show-code -->\n<template>\n  <PnVTK\n    :object=\"vtkjs_url.value\" \n    sizing_mode=\"stretch_width\" \n    :height=\"400\" \n    :enable_keybindings=\"True\" \n    :orientation_widget=\"True\"\n  />\n  <PnButton @click=\"update_object()\">\u66f4\u6362 3D \u6a21\u578b</PnButton>\n</template>\n<script lang='py'>\nfrom vuepy import ref\n\nvtkjs_url = ref(\"https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs\")\n\ndef update_object():\n    vtkjs_url.value = \"https://raw.githubusercontent.com/Kitware/vtk-js-datasets/master/data/vtkjs/TBarAssembly.vtkjs\"\n</script>\n", "setup": ""}



## 相机控制

一旦显示了 VTKJS 组件，它将自动将相机状态与组件对象同步。相机参数仅在交互结束时更新。我们可以在相应的参数上读取相机状态：


In [3]:
%%vuepy_run --plugins vpanel --show-code --codegen-backend='panel'
<template>
  <PnVTK 
    object="https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs"
    :sizing_mode="'stretch_width'" 
    :height="400" 
    :enable_keybindings="True" 
    :orientation_widget="True"
    ref="vtk_pane_ref" />
  <PnButton @click="read_camera()">读取相机状态</PnButton>
  <PnJSON v-if="camera_state.value" :object="camera_state.value" :depth="1" />
</template>
<script lang='py'>
from vuepy import ref

vtk_pane_ref = ref(None)
camera_state = ref(None)

def read_camera():
    vtk_pane = vtk_pane_ref.value.unwrap()
    if vtk_pane.camera:
        camera_state.value = vtk_pane.camera
</script>

{"vue": "<!-- --plugins vpanel --show-code --codegen-backend='panel' -->\n<template>\n  <PnVTK \n    object=\"https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs\"\n    :sizing_mode=\"'stretch_width'\" \n    :height=\"400\" \n    :enable_keybindings=\"True\" \n    :orientation_widget=\"True\"\n    ref=\"vtk_pane_ref\" />\n  <PnButton @click=\"read_camera()\">\u8bfb\u53d6\u76f8\u673a\u72b6\u6001</PnButton>\n  <PnJSON v-if=\"camera_state.value\" :object=\"camera_state.value\" :depth=\"1\" />\n</template>\n<script lang='py'>\nfrom vuepy import ref\n\nvtk_pane_ref = ref(None)\ncamera_state = ref(None)\n\ndef read_camera():\n    vtk_pane = vtk_pane_ref.value.unwrap()\n    if vtk_pane.camera:\n        camera_state.value = vtk_pane.camera\n</script>\n", "setup": ""}



这种技术也使得可以将两个或多个 VTKJS 组件的相机链接在一起：

还可以在 Python 中修改相机状态并触发更新：

In [4]:
%%vuepy_run --plugins vpanel --show-code --codegen-backend='panel'
<template>
  <PnRow>
    <PnVTK 
      object="https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs"
      :height="400" 
      :sizing_mode="'stretch_width'"
      ref="dragon1_ref" />
    <PnVTK 
      object="https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs"
      :height="400" 
      :sizing_mode="'stretch_width'"
      ref="dragon2_ref" />
  </PnRow>
  <PnButton @click="change_view_angle()">改变视角</PnButton>
</template>
<script lang='py'>
from vuepy import ref, onMounted

dragon1_ref = ref(None)
dragon2_ref = ref(None)

@onMounted
def on_render():
    dragon1 = dragon1_ref.value.unwrap()
    dragon2 = dragon2_ref.value.unwrap()
    # 双向链接两个组件的相机
    dragon1.jslink(dragon2, camera='camera', bidirectional=True)
    
def change_view_angle():
    dragon1 = dragon1_ref.value.unwrap()
    if dragon1.camera:
        dragon1.camera['viewAngle'] = 50
        dragon1.param.trigger('camera')
</script>

{"vue": "<!-- --plugins vpanel --show-code --codegen-backend='panel' -->\n<template>\n  <PnRow>\n    <PnVTK \n      object=\"https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs\"\n      :height=\"400\" \n      :sizing_mode=\"'stretch_width'\"\n      ref=\"dragon1_ref\" />\n    <PnVTK \n      object=\"https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs\"\n      :height=\"400\" \n      :sizing_mode=\"'stretch_width'\"\n      ref=\"dragon2_ref\" />\n  </PnRow>\n  <PnButton @click=\"change_view_angle()\">\u6539\u53d8\u89c6\u89d2</PnButton>\n</template>\n<script lang='py'>\nfrom vuepy import ref, onMounted\n\ndragon1_ref = ref(None)\ndragon2_ref = ref(None)\n\n@onMounted\ndef on_render():\n    dragon1 = dragon1_ref.value.unwrap()\n    dragon2 = dragon2_ref.value.unwrap()\n    # \u53cc\u5411\u94fe\u63a5\u4e24\u4e2a\u7ec4\u4ef6\u7684\u76f8\u673a\n    dragon1.jslink(dragon2, camera='camera', bidirectional=True)\n    \ndef change_view_


## API

### 属性

| 属性名                      | 说明                          | 类型                                                           | 默认值 |
| -------------------------- | ----------------------------- | ---------------------------------------------------------------| ------- |
| object                     | 可以是指向本地或远程的带有 `.vtkjs` 扩展名的文件的字符串 | ^[str, object]                  | None |
| axes                       | 在 3D 视图中构造的坐标轴的参数字典。必须至少包含 `xticker`、`yticker` 和 `zticker` | ^[dict]    | None |
| camera                     | 反映 VTK 相机当前状态的字典      | ^[dict]                                                       | None |
| enable_keybindings         | 激活/禁用键盘绑定的布尔值。绑定的键有：s（将所有 actor 表示设置为*表面*）、w（将所有 actor 表示设置为*线框*）、v（将所有 actor 表示设置为*顶点*）、r（居中 actor 并移动相机，使所有 actor 可见） | ^[boolean] | False |
| orientation_widget         | 激活/禁用 3D 面板中的方向部件的布尔值 | ^[boolean]                                                  | False |
| interactive_orientation_widget | 如果为 True，则方向部件可点击并允许将场景旋转到正交投影之一 | ^[boolean]                | False |
| sizing_mode                | 尺寸调整模式                   | ^[str]                                                         | 'fixed'  |
| width                      | 宽度                          | ^[int, str]                                                    | None    |
| height                     | 高度                          | ^[int, str]                                                    | None    |
| min_width                  | 最小宽度                      | ^[int]                                                         | None    |
| min_height                 | 最小高度                      | ^[int]                                                         | None    |
| max_width                  | 最大宽度                      | ^[int]                                                         | None    |
| max_height                 | 最大高度                      | ^[int]                                                         | None    |
| margin                     | 外边距                        | ^[int, tuple]                                                  | 5       |
| css_classes                | CSS类名列表                   | ^[list]                                                        | []      |
### Slots

| 插槽名   | 说明               |
| ---     | ---               |
| default | 自定义默认内容      |

### 方法

| 方法名 | 说明 | 参数 |
| --- | --- | --- |
| export_scene | 导出场景并生成可以被官方 vtk-js 场景导入器加载的文件 | filename: str |


## Controls

In [5]:
##controls
import panel as pn
pn.extension('vtk')

vtk_pane = pn.pane.VTK(
    'https://raw.githubusercontent.com/Kitware/vtk-js/master/Data/StanfordDragon.vtkjs',
    sizing_mode='stretch_width', height=400, enable_keybindings=True, orientation_widget=True
)
pn.Row(vtk_pane.controls(jslink=True), vtk_pane)

Row
    [0] Tabs
        [0] WidgetBox(margin=(5, 10), name='Controls')
            [0] StaticText(value='<b>Controls</b>')
            [1] DictInput(description="Parameters of the axes to..., name='Axes', serializer='json', type=<class 'dict'>)
            [2] DictInput(description='State of the r..., name='Camera', serializer='json', type=<class 'dict'>)
            [3] ListInput(description='Color mapper o..., name='Color mappers', serializer='json', type=<class 'list'>)
            [4] Checkbox(name='Orientation widget', value=True)
            [5] Checkbox(name='Interactive o..., value=True)
            [6] Checkbox(name='Enable keybindings', value=True)
        [1] WidgetBox(margin=(5, 10), name='Layout')
            [0] StaticText(value='<b>Layout</b>')
            [1] TextInput(description='String identifier f..., name='Name', value='VTKJS00155')
            [2] LiteralInput(description='Whether the object should..., name='Align', serializer='json', value='start')
            [3] LiteralInput(description='Describes the proportiona..., name='Aspect ratio', serializer='json')
            [4] ListInput(description='CSS classes t..., name='Css classes', serializer='json', type=<class 'list'>)
            [5] IntInput(description='The height of the compone..., name='Height', start=0, value=400)
            [6] IntInput(description='Minimal width o..., name='Min width', start=0)
            [7] IntInput(description='Minimal height o..., name='Min height', start=0)
            [8] IntInput(description='Maximum width o..., name='Max width', start=0)
            [9] IntInput(description='Maximum height o..., name='Max height', start=0)
            [10] LiteralInput(description='Allows to create addition..., name='Margin', serializer='json', value=(5, 10))
            [11] DictInput(description='Dictionary of C..., name='Styles', serializer='json', type=<class 'dict'>)
            [12] ListInput(description='List of arbitrary t..., name='Tags', serializer='json', type=<class 'list'>)
            [13] IntInput(description='The width of the componen..., name='Width', start=0)
            [14] Select(description='Describes how the compone..., name='Width policy', options=OrderedDict([('auto', ...]), value='auto')
            [15] Select(description='Describes how the compone..., name='Height policy', options=OrderedDict([('auto', ...]), value='auto')
            [16] Select(description='How the component should ..., name='Sizing mode', options=OrderedDict([('fixed', ...]), value='stretch_width')
            [17] Checkbox(name='Visible', value=True)
    [1] VTKJS(str, enable_keybindings=True, height=400, orientation_widget=True, sizing_mode='stretch_width')